## Incrementally update a drilling campaign object with interim data

This example is based on the update notebook, but it publishes interim data one snapshot at a time from `sample-data/interim-progressive`. The next snapshot index is stored in `notebook-data/.env`, so each time you run the publish cell the notebook advances from `interim_00.csv` to `interim_01.csv`, and so on.

> **Note:** This notebook requires an existing drilling campaign object in your Evo workspace with a populated `planned` section.
>
> The first snapshot contains no interim data. Publishing it will remove any existing `interim` section from the selected object.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameter must be provided:

- The client ID of your Evo application.

To obtain app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [ ]:
import pandas as pd
from evo.notebooks import ServiceManagerWidget

cache_location = "./notebook-data"
input_path = "sample-data"
progress_input_path = f"{input_path}/interim-progressive"
object_hole_id = "Holeid"

# Evo app credentials
client_id = "<your_client_id>"  # Replace with your client ID

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    cache_location=cache_location,
).login()

In [ ]:
import asyncio
import copy
import uuid
from datetime import datetime
from pathlib import Path
from uuid import UUID

import ipywidgets as widgets
import numpy as np
from dotenv import dotenv_values, set_key
from IPython.display import JSON, display

from evo.notebooks import FeedbackWidget
from evo.objects import ObjectAPIClient
from evo_schemas.components import (
    CategoryAttribute_V1_1_0,
    DownholeDirectionVector_V1_0_0,
    HoleChunks_V1_0_0,
    HoleCollars_V1_0_0,
    NanCategorical_V1_0_1,
    StringAttribute_V1_1_0,
)
from evo_schemas.elements import FloatArray3_V1_0_1, IntegerArray1_V1_0_1, LookupTable_V1_0_1, StringArray_V1_0_1
from evo_schemas.objects import DrillingCampaign_V1_0_0_Interim

object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())
data_client = object_client.get_data_client(manager.cache)

progress_dir = Path(progress_input_path)
state_file = Path(cache_location) / ".env"
counter_key = "INTERIM_PROGRESS_INDEX"


def create_hole_id_mapping(hole_id_table, value_list):
    num_keys = len(hole_id_table.index)

    mapping_df = pd.DataFrame(list())
    mapping_df["hole_index"] = hole_id_table["key"]
    mapping_df["offset"] = [0] * num_keys
    mapping_df["count"] = [0] * num_keys

    mapping_df["hole_index"] = mapping_df["hole_index"].astype("int32")
    mapping_df["offset"] = mapping_df["offset"].astype("uint64")
    mapping_df["count"] = mapping_df["count"].astype("uint64")

    prev_value = ""
    key = ""
    count = 0
    offset = 0

    for _, row in value_list.iterrows():
        new_value = row["data"]

        if new_value != prev_value:
            if prev_value != "":
                mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
                mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset
                offset += count

            mask = hole_id_table["value"] == new_value
            masked_df = hole_id_table[mask]
            try:
                key_row = masked_df.iloc[[0]]
            except IndexError:
                print("Ignoring this hole ID")
                continue

            key = key_row["key"].iloc[0]
            count = 1
            prev_value = new_value
        else:
            count += 1

    mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
    mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset

    return mapping_df


def create_category_lookup_and_values(attribute):
    attribute = attribute.copy()
    attribute.replace(np.nan, "", regex=True, inplace=True)
    list_obj = sorted(set(attribute["data"]))

    table_df = pd.DataFrame([])
    table_df["key"] = list(range(0, len(list_obj)))
    table_df["value"] = list_obj

    values_df = pd.DataFrame([])
    values_df["data"] = attribute["data"].map(table_df.set_index("value")["key"])
    return table_df, values_df


def serialize_for_display(obj):
    if isinstance(obj, UUID):
        return str(obj)
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, list):
        return [serialize_for_display(item) for item in obj]
    if isinstance(obj, dict):
        return {key: serialize_for_display(value) for key, value in obj.items()}
    if hasattr(obj, "__dict__"):
        return {key: serialize_for_display(value) for key, value in vars(obj).items()}
    return obj


async def download_table(table_info, label, object_id):
    return (
        await data_client.download_table(
            object_id=object_id,
            version_id=None,
            table_info=table_info,
            fb=FeedbackWidget(label),
        )
    ).to_pandas()


def ensure_state_file():
    state_file.parent.mkdir(parents=True, exist_ok=True)
    if not state_file.exists():
        state_file.write_text("")


def read_progress_index():
    ensure_state_file()
    raw_value = dotenv_values(state_file).get(counter_key)
    if raw_value in (None, ""):
        set_key(str(state_file), counter_key, "0")
        return 0

    try:
        index = int(raw_value)
    except ValueError as exc:
        raise ValueError(f"Invalid {counter_key} value in {state_file}: {raw_value!r}") from exc

    if index < 0:
        raise ValueError(f"{counter_key} must be non-negative, got {index}.")
    return index


def write_progress_index(index):
    ensure_state_file()
    set_key(str(state_file), counter_key, str(index))


def get_snapshot_files():
    snapshot_files = sorted(progress_dir.glob("interim_*.csv"))
    if not snapshot_files:
        raise FileNotFoundError(f"No interim snapshot CSVs found in {progress_dir}")
    return snapshot_files


def get_current_snapshot():
    snapshot_files = get_snapshot_files()
    snapshot_index = read_progress_index()

    if snapshot_index >= len(snapshot_files):
        raise ValueError(
            f"All interim snapshots have already been published. Reset {counter_key} in {state_file} to 0 to start over."
        )

    return snapshot_index, snapshot_files[snapshot_index], snapshot_files


async def resolve_selected_snapshot(selected_object_id):
    downloaded_object = await object_client.download_object_by_id(object_id=selected_object_id, version=None)
    metadata = downloaded_object.metadata
    downloaded_dict = downloaded_object.as_dict()
    snapshot_index, snapshot_file, snapshot_files = get_current_snapshot()
    return downloaded_object, metadata, downloaded_dict, snapshot_index, snapshot_file, snapshot_files


async def build_interim_component_dict(downloaded_dict, metadata, snapshot_file):
    if "planned" not in downloaded_dict:
        raise ValueError("The selected drilling campaign does not contain a planned section.")

    planned_collar = downloaded_dict["planned"]["collar"]
    hole_id_table_df = await download_table(
        downloaded_dict["hole_id"]["table"],
        "Downloading hole ID lookup table",
        metadata.id,
    )

    coordinates_component = FloatArray3_V1_0_1.from_dict(planned_collar["coordinates"])
    distances_component = FloatArray3_V1_0_1.from_dict(planned_collar["distances"])

    interim_df = pd.read_csv(snapshot_file)
    interim_df = interim_df.sort_values([object_hole_id]).reset_index(drop=True)
    interim_df = interim_df[interim_df[object_hole_id].isin(hole_id_table_df["value"])].reset_index(drop=True)

    interim_component_dict = None

    if interim_df.empty:
        print("The current snapshot contains no interim data. The published object will not include an interim section.")
        return interim_component_dict

    survey_attributes = [
        {
            "name": "Wolfpass GM",
            "key": str(uuid.uuid4()),
            "type": "category",
        }
    ]

    survey_hole_values_df = pd.DataFrame({"data": interim_df[object_hole_id]})
    interim_hole_chunks_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=survey_hole_values_df)
    interim_hole_chunks_component = HoleChunks_V1_0_0.from_dict(data_client.save_dataframe(interim_hole_chunks_df))

    survey_distances_df = interim_df[["depth", "azimuth", "dip"]].copy()
    survey_distances_df = survey_distances_df.rename(columns={"depth": "distance"})
    survey_distances_df = survey_distances_df.astype(float).reset_index(drop=True)
    interim_survey_component = DownholeDirectionVector_V1_0_0.from_dict(data_client.save_dataframe(survey_distances_df))

    interim_attributes = []
    for survey_attribute in survey_attributes:
        attribute_name = survey_attribute["name"]
        attribute_key = survey_attribute["key"]
        attribute_type = survey_attribute["type"]

        attribute_df = pd.DataFrame({"data": interim_df[attribute_name]})

        if attribute_type == "string":
            attribute_df = attribute_df.astype(str).reset_index(drop=True)
            values = StringArray_V1_0_1.from_dict(data_client.save_dataframe(attribute_df))
            attribute = StringAttribute_V1_1_0(name=attribute_name, key=attribute_key, values=values)
        elif attribute_type == "category":
            attribute_df = attribute_df.astype(str).reset_index(drop=True)
            table_df, values_df = create_category_lookup_and_values(attribute_df)

            table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
            values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))
            attribute = CategoryAttribute_V1_1_0(
                name=attribute_name,
                key=attribute_key,
                nan_description=NanCategorical_V1_0_1(values=[]),
                table=table,
                values=values,
            )
        else:
            raise ValueError(f"Unsupported attribute type: {attribute_type}")

        interim_attributes.append(attribute)

    interim_hole_collars_component = HoleCollars_V1_0_0(
        coordinates=coordinates_component,
        distances=distances_component,
        holes=interim_hole_chunks_component,
        attributes=interim_attributes,
    )

    interim_component = DrillingCampaign_V1_0_0_Interim(
        collar=interim_hole_collars_component,
        path=interim_survey_component,
    )
    interim_component_dict = interim_component.as_dict()
    return interim_component_dict


async def publish_snapshot_once(selected_object_id):
    _, metadata, downloaded_dict, snapshot_index, snapshot_file, snapshot_files = await resolve_selected_snapshot(selected_object_id)
    interim_component_dict = await build_interim_component_dict(downloaded_dict, metadata, snapshot_file)

    latest_downloaded_object = await object_client.download_object_by_id(object_id=selected_object_id, version=None)
    latest_downloaded_dict = latest_downloaded_object.as_dict()

    updated_object_dict = copy.deepcopy(latest_downloaded_dict)
    updated_object_dict["planned"] = latest_downloaded_dict["planned"]

    if interim_component_dict is None:
        updated_object_dict.pop("interim", None)
    else:
        updated_object_dict["interim"] = interim_component_dict

    await data_client.upload_referenced_data(updated_object_dict, FeedbackWidget(f"Uploading snapshot {snapshot_index:02d}"))
    updated_object = await latest_downloaded_object.update(updated_object_dict)

    next_snapshot_index = snapshot_index + 1
    write_progress_index(next_snapshot_index)

    result = {
        "updated_object": updated_object,
        "snapshot_index": snapshot_index,
        "snapshot_file": snapshot_file,
        "snapshot_files": snapshot_files,
        "next_snapshot_index": next_snapshot_index,
    }

    print(f"Successfully published snapshot {snapshot_index:02d} from {snapshot_file.name}.")
    print(f"New version ID: {updated_object.metadata.version_id}")

    if next_snapshot_index < len(snapshot_files):
        print(f"Next run will publish {snapshot_files[next_snapshot_index].name}.")
    else:
        print(f"All {len(snapshot_files)} snapshots have been published. Reset {counter_key} in {state_file} to 0 to start over.")

    return result


async def publish_snapshots_on_timer(selected_object_id, interval_seconds=10, iterations=None):
    run_count = 0

    while True:
        result = await publish_snapshot_once(selected_object_id)
        run_count += 1

        if iterations is not None and run_count >= iterations:
            print(f"Stopped after {run_count} timed publish run(s).")
            return

        if result["next_snapshot_index"] >= len(result["snapshot_files"]):
            return

        print(f"Waiting {interval_seconds} seconds before the next publish...")
        await asyncio.sleep(interval_seconds)

### Load all drilling campaigns in the workspace

The next cell lists drilling campaign objects in the currently selected Evo workspace and displays a dropdown so you can choose one to update.

In [ ]:
all_objects = await object_client.list_all_objects()
drilling_campaigns = [obj for obj in all_objects if "drilling-campaign" in obj.schema_id.sub_classification]

if len(drilling_campaigns) == 0:
    print("No drilling campaigns found in the selected workspace.")
else:
    dropdown_options = [(obj.name.removesuffix(".json"), str(obj.id)) for obj in drilling_campaigns]

    drilling_campaign_dropdown = widgets.Dropdown(
        options=dropdown_options,
        description="Drilling campaign:",
        disabled=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )

    print(f"Found {len(drilling_campaigns)} drilling campaign(s). Please select one:")
    display(drilling_campaign_dropdown)

### Download the selected drilling campaign and resolve the next snapshot

This cell downloads the selected drilling campaign object and resolves the next progressive interim CSV based on the counter stored in `notebook-data/.env`.

In [ ]:
selected_object_id = drilling_campaign_dropdown.value
if selected_object_id is None:
    raise ValueError("Select a drilling campaign before running this cell.")

downloaded_object = await object_client.download_object_by_id(object_id=selected_object_id, version=None)
metadata = downloaded_object.metadata
downloaded_dict = downloaded_object.as_dict()
snapshot_index, snapshot_file, snapshot_files = get_current_snapshot()

display(JSON(serialize_for_display(downloaded_dict), expanded=False))
print(f"Selected object: {downloaded_dict['name']}")
print(f"Object ID: {metadata.id}")
print(f"Current version: {metadata.version_id}")
print(f"Publishing snapshot {snapshot_index:02d}: {snapshot_file.name}")
print(f"State file: {state_file}")

In [ ]:
from evo_schemas.components import (
    CategoryAttribute_V1_1_0,
    DownholeDirectionVector_V1_0_0,
    HoleChunks_V1_0_0,
    HoleCollars_V1_0_0,
    NanCategorical_V1_0_1,
    StringAttribute_V1_1_0,
)
from evo_schemas.elements import FloatArray3_V1_0_1, IntegerArray1_V1_0_1, LookupTable_V1_0_1, StringArray_V1_0_1
from evo_schemas.objects import DrillingCampaign_V1_0_0_Interim

if "planned" not in downloaded_dict:
    raise ValueError("The selected drilling campaign does not contain a planned section.")

planned_section = copy.deepcopy(downloaded_dict["planned"])
planned_collar = planned_section["collar"]
hole_id_table_df = await download_table(
    downloaded_dict["hole_id"]["table"],
    "Downloading hole ID lookup table",
    metadata.id,
)

coordinates_component = FloatArray3_V1_0_1.from_dict(planned_collar["coordinates"])
distances_component = FloatArray3_V1_0_1.from_dict(planned_collar["distances"])

interim_df = pd.read_csv(snapshot_file)
interim_df = interim_df.sort_values([object_hole_id]).reset_index(drop=True)
interim_df = interim_df[interim_df[object_hole_id].isin(hole_id_table_df["value"])].reset_index(drop=True)

interim_component_dict = None

if interim_df.empty:
    print("The current snapshot contains no interim data. The published object will not include an interim section.")
else:
    survey_attributes = [
        {
            "name": "Wolfpass GM",
            "key": str(uuid.uuid4()),
            "type": "category",
        }
    ]

    survey_hole_values_df = pd.DataFrame({"data": interim_df[object_hole_id]})
    interim_hole_chunks_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=survey_hole_values_df)
    interim_hole_chunks_component = HoleChunks_V1_0_0.from_dict(data_client.save_dataframe(interim_hole_chunks_df))

    survey_distances_df = interim_df[["depth", "azimuth", "dip"]].copy()
    survey_distances_df = survey_distances_df.rename(columns={"depth": "distance"})
    survey_distances_df = survey_distances_df.astype(float).reset_index(drop=True)
    interim_survey_component = DownholeDirectionVector_V1_0_0.from_dict(data_client.save_dataframe(survey_distances_df))

    interim_attributes = []
    for survey_attribute in survey_attributes:
        attribute_name = survey_attribute["name"]
        attribute_key = survey_attribute["key"]
        attribute_type = survey_attribute["type"]

        attribute_df = pd.DataFrame({"data": interim_df[attribute_name]})

        if attribute_type == "string":
            attribute_df = attribute_df.astype(str).reset_index(drop=True)
            values = StringArray_V1_0_1.from_dict(data_client.save_dataframe(attribute_df))
            attribute = StringAttribute_V1_1_0(name=attribute_name, key=attribute_key, values=values)
        elif attribute_type == "category":
            attribute_df = attribute_df.astype(str).reset_index(drop=True)
            table_df, values_df = create_category_lookup_and_values(attribute_df)

            table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
            values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))
            attribute = CategoryAttribute_V1_1_0(
                name=attribute_name,
                key=attribute_key,
                nan_description=NanCategorical_V1_0_1(values=[]),
                table=table,
                values=values,
            )
        else:
            raise ValueError(f"Unsupported attribute type: {attribute_type}")

        interim_attributes.append(attribute)

    interim_hole_collars_component = HoleCollars_V1_0_0(
        coordinates=coordinates_component,
        distances=distances_component,
        holes=interim_hole_chunks_component,
        attributes=interim_attributes,
    )

    interim_component = DrillingCampaign_V1_0_0_Interim(
        collar=interim_hole_collars_component,
        path=interim_survey_component,
    )
    interim_component_dict = interim_component.as_dict()

interim_component_dict

In [ ]:
latest_downloaded_object = await object_client.download_object_by_id(object_id=selected_object_id, version=None)
latest_downloaded_dict = latest_downloaded_object.as_dict()

updated_object_dict = copy.deepcopy(latest_downloaded_dict)
updated_object_dict["planned"] = latest_downloaded_dict["planned"]

if interim_component_dict is None:
    updated_object_dict.pop("interim", None)
else:
    updated_object_dict["interim"] = interim_component_dict

await data_client.upload_referenced_data(updated_object_dict, FeedbackWidget(f"Uploading snapshot {snapshot_index:02d}"))
updated_object = await latest_downloaded_object.update(updated_object_dict)

downloaded_object = updated_object
downloaded_dict = updated_object.as_dict()
metadata = updated_object.metadata

next_snapshot_index = snapshot_index + 1
write_progress_index(next_snapshot_index)

print(f"Successfully published snapshot {snapshot_index:02d} from {snapshot_file.name}.")
print(f"New version ID: {updated_object.metadata.version_id}")

if next_snapshot_index < len(snapshot_files):
    print(f"Next run will publish {snapshot_files[next_snapshot_index].name}.")
    print("Before publishing the next snapshot, rerun cells 7 and 8 to reload the next snapshot payload.")
else:
    print(f"All {len(snapshot_files)} snapshots have been published. Reset {counter_key} in {state_file} to 0 to start over.")

### Optional: publish snapshots on a timer

If you want to simulate real-time publishing, run the next cell after selecting a drilling campaign. It will publish one snapshot every 10 seconds until you stop it or it runs out of snapshots.

In [ ]:
interval_seconds = 30
iterations = None  # Set to an integer to limit the number of publishes.

selected_object_id = drilling_campaign_dropdown.value
if selected_object_id is None:
    raise ValueError("Select a drilling campaign before running this cell.")

print(f"Starting timed publishing every {interval_seconds} seconds.")
print("Interrupt the kernel to stop early.")

await publish_snapshots_on_timer(
    selected_object_id=selected_object_id,
    interval_seconds=interval_seconds,
    iterations=iterations,
)

## Summary

In this example, you:

- Logged in and selected an Evo instance, workspace, and drilling campaign object.
- Resolved the next interim snapshot from `sample-data/interim-progressive`.
- Read the current snapshot index from `notebook-data/.env`.
- Published one incremental update to the drilling campaign object.
- Advanced the counter so the next run publishes the next snapshot.